#Nhóm 4 – Nhận diện ký tự biển số xe bằng KNN

Notebook này trình bày toàn bộ pipeline:
1. Gán nhãn ảnh ký tự (`init-labels`)
2. Huấn luyện mô hình KNN (`train`)
3. Đánh giá & xuất báo cáo
4. Dự đoán ký tự từ thư mục ảnh mới (`predict-folder`)


## 1. Import thư viện & khai báo hằng số

In [1]:
import csv
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np

# ── Hằng số cấu hình ─────────────────────────────────────────────────────────
CHAR_SIZE          = (20, 30)           # width × height của mỗi ký tự
DEFAULT_CHAR_DIR   = Path("data/char_images")
DEFAULT_MODEL_DIR  = Path("model_data")

LABELS_CSV          = "labels.csv"
CLASSIFICATIONS_TXT = "classifications.txt"
FLATTENED_IMAGES_TXT= "flattened_images.txt"
KNN_MODEL           = "knn_model.yml"
EVALUATION_REPORT   = "evaluation_report.txt"
CONFUSION_MATRIX    = "confusion_matrix.csv"

print("Import xong.")


Import xong.


## 2. Tiền xử lý ảnh ký tự

Mỗi ảnh ký tự được:
- Đọc dưới dạng **grayscale**
- **Resize** về `20×30` px
- Làm mịn bằng **GaussianBlur**
- **Nhị phân hoá** bằng Otsu threshold
- **Flatten** thành vector 600 chiều (20 × 30) cho KNN


In [2]:
def iter_char_images(char_dir=DEFAULT_CHAR_DIR):
    """Duyệt tất cả ảnh char_*.png trong các thư mục con."""
    return sorted(
        p for p in Path(char_dir).glob("*/*.png")
        if p.is_file() and p.name.startswith("char_")
    )


def preprocess_char(image_path):
    """Đọc & tiền xử lý 1 ảnh ký tự → vector float32 (1, 600)."""
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Không đọc được ảnh: {image_path}")

    img = cv2.resize(img, CHAR_SIZE)
    img = cv2.GaussianBlur(img, (3, 3), 0)

    _, binary = cv2.threshold(img, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary.reshape(1, -1).astype(np.float32)


print("Hàm tiền xử lý đã sẵn sàng.")


Hàm tiền xử lý đã sẵn sàng.


## 3. Bước 1 – Tạo / cập nhật file nhãn (`labels.csv`)

Chạy cell này **một lần** để sinh file `labels.csv`.  
Sau đó **mở file, điền cột `label`** bằng ký tự thật (0-9, A-Z) rồi mới train.


In [ ]:
def init_labels(char_dir=DEFAULT_CHAR_DIR, model_dir=DEFAULT_MODEL_DIR):
    model_dir = Path(model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    labels_path = model_dir / LABELS_CSV

    # Nếu file đã tồn tại thì chỉ load, không ghi đè
    if labels_path.exists():
        print(f"Đã có sẵn file {labels_path}, giữ nguyên nội dung.")
        with labels_path.open("r", newline="", encoding="utf-8") as f:
            rows = list(csv.DictReader(f))
        print(f"Đã load {len(rows)} nhãn từ file.")
        return rows

    # Nếu chưa có thì mới tạo mới
    rows = []
    for image_path in iter_char_images(char_dir):
        key = image_path.as_posix()
        rows.append({"image_path": key, "label": ""})

    with labels_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image_path", "label"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"Đã tạo mới {labels_path} với {len(rows)} ảnh ký tự.")
    print("Hãy điền cột 'label' bằng ký tự thật: 0-9, A-Z.")
    return rows



# ── Chạy ─────────────────────────────────────────────────────────────────────
init_labels(model_dir="E:/Nhom A/model_data")



Đã tạo/cập nhật E:\Nhom A\model_data\labels.csv với 0 ảnh ký tự.
Hãy điền cột 'label' bằng ký tự thật: 0-9, A-Z.


## 4. Load dữ liệu đã gán nhãn

Đọc `labels.csv`, tiền xử lý từng ảnh và tạo ma trận đặc trưng `X` + vector nhãn `y`.


In [3]:
def load_labeled_samples(model_dir=DEFAULT_MODEL_DIR):
    labels_path = Path(model_dir) / LABELS_CSV
    if not labels_path.exists():
        raise FileNotFoundError(
            f"Chưa có {labels_path}. Hãy chạy init_labels() trước."
        )

    samples, labels, paths = [], [], []

    with labels_path.open("r", newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            label = row.get("label", "").strip().upper()
            image_path = Path(row["image_path"])

            if not label:
                continue
            if len(label) != 1:
                raise ValueError(
                    f"Nhãn phải là 1 ký tự tại {image_path}: '{label}'"
                )

            samples.append(preprocess_char(image_path))
            labels.append(ord(label))
            paths.append(image_path)

    if not samples:
        raise ValueError("Chưa có ảnh nào được gán nhãn trong labels.csv.")

    X = np.vstack(samples).astype(np.float32)
    y = np.array(labels, dtype=np.float32).reshape(-1, 1)
    print(f"Load xong: {len(samples)} mẫu, {len(set(labels))} lớp ký tự.")
    return X, y, paths


X, y, paths = load_labeled_samples()


FileNotFoundError: Chưa có model_data\labels.csv. Hãy chạy init_labels() trước.

## 5. Chia tập Train / Test (Stratified)

Dùng **stratified split** để đảm bảo mỗi lớp ký tự đều xuất hiện ở cả tập train lẫn test.


In [ ]:
def stratified_split(y, test_size=0.25, seed=42):
    rng = np.random.default_rng(seed)
    groups = defaultdict(list)

    for idx, label in enumerate(y.ravel().astype(int)):
        groups[label].append(idx)

    train_idx, test_idx = [], []
    for _, indices in groups.items():
        indices = np.array(indices)
        rng.shuffle(indices)

        if len(indices) == 1:          # chỉ có 1 mẫu → để vào train
            train_idx.extend(indices.tolist())
            continue

        n_test = max(1, int(round(len(indices) * test_size)))
        n_test = min(n_test, len(indices) - 1)
        test_idx.extend(indices[:n_test].tolist())
        train_idx.extend(indices[n_test:].tolist())

    return np.array(train_idx), np.array(test_idx)


# ── Cấu hình ─────────────────────────────────────────────────────────────────
TEST_SIZE = 0.25
SEED      = 42

train_idx, test_idx = stratified_split(y, TEST_SIZE, SEED)
print(f"Train: {len(train_idx)} mẫu | Test: {len(test_idx)} mẫu")


## 6. Huấn luyện mô hình KNN

Sử dụng `cv2.ml.KNearest` – KNN tích hợp sẵn trong OpenCV.  
Tham số `K` có thể điều chỉnh ở cell dưới.


In [ ]:
K = 3   # ← đổi K tại đây nếu muốn thử nghiệm


def train_knn(X_train, y_train, k=K):
    knn = cv2.ml.KNearest_create()
    knn.setDefaultK(k)
    knn.train(X_train, cv2.ml.ROW_SAMPLE, y_train)
    return knn


# Train trên tập con để đánh giá
knn_eval = train_knn(X[train_idx], y[train_idx], K)
print(f"✅ Đã huấn luyện KNN (K={K}) trên {len(train_idx)} mẫu.")


## 7. Đánh giá mô hình trên tập Test


In [ ]:
def evaluate(knn, X_test, y_test, k=K):
    if len(X_test) == 0:
        return None, []
    _, results, _, _ = knn.findNearest(X_test, k=k)
    y_true = y_test.ravel().astype(int)
    y_pred = results.ravel().astype(int)
    accuracy = float(np.mean(y_true == y_pred))
    rows = [(chr(t), chr(p)) for t, p in zip(y_true, y_pred)]
    return accuracy, rows


accuracy, confusion_rows = evaluate(knn_eval, X[test_idx], y[test_idx], K)
print(f"🎯 Accuracy trên tập test: {accuracy:.4f}  ({accuracy*100:.1f}%)")

errors = [(t, p) for t, p in confusion_rows if t != p]
if errors:
    print(f"⚠️  Số mẫu nhận diện sai: {len(errors)}")
    for true_c, pred_c in errors:
        print(f"   Thật: {true_c}  →  Dự đoán: {pred_c}")
else:
    print("✅ Không có mẫu nào bị nhận diện sai trên tập test!")


## 8. Hiển thị Confusion Matrix


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

labels_set = sorted({x for row in confusion_rows for x in row})
n = len(labels_set)
label2idx = {l: i for i, l in enumerate(labels_set)}
matrix = np.zeros((n, n), dtype=int)

for true_c, pred_c in confusion_rows:
    matrix[label2idx[true_c]][label2idx[pred_c]] += 1

fig, ax = plt.subplots(figsize=(max(6, n * 0.55), max(5, n * 0.5)))
im = ax.imshow(matrix, cmap="Blues")
ax.set_xticks(range(n)); ax.set_xticklabels(labels_set)
ax.set_yticks(range(n)); ax.set_yticklabels(labels_set)
ax.set_xlabel("Dự đoán"); ax.set_ylabel("Thực tế")
ax.set_title(f"Confusion Matrix – KNN (K={K})  |  Accuracy: {accuracy:.2%}")
plt.colorbar(im, ax=ax)

for i in range(n):
    for j in range(n):
        if matrix[i, j]:
            ax.text(j, i, matrix[i, j], ha="center", va="center",
                    color="white" if matrix[i, j] > matrix.max() * 0.6 else "black",
                    fontsize=8)
plt.tight_layout()
plt.show()


## 9. Lưu model & xuất báo cáo

Train lại trên **toàn bộ** dữ liệu có nhãn rồi lưu `.yml`.  
Đồng thời xuất `evaluation_report.txt` và `confusion_matrix.csv`.


In [ ]:
def write_evaluation(model_dir, accuracy, confusion_rows, label_counts, k):
    model_dir = Path(model_dir)
    report_path = model_dir / EVALUATION_REPORT
    matrix_path = model_dir / CONFUSION_MATRIX

    with report_path.open("w", encoding="utf-8") as f:
        f.write("Báo cáo đánh giá KNN – Nhóm 4\n")
        f.write(f"K = {k}\n")
        f.write(f"Số mẫu đã gán nhãn = {sum(label_counts.values())}\n")
        f.write("Phân bố nhãn:\n")
        for label, count in sorted(label_counts.items()):
            f.write(f"  {label}: {count}\n")
        if accuracy is None:
            f.write("Chưa đủ mẫu test để tính accuracy.\n")
        else:
            f.write(f"Accuracy = {accuracy:.4f}\n")
            errors = [(t, p) for t, p in confusion_rows if t != p]
            f.write(f"Số mẫu sai = {len(errors)}\n")
            for true_c, pred_c in errors:
                f.write(f"  {true_c} → {pred_c}\n")

    # Confusion matrix CSV
    labels_set = sorted({x for row in confusion_rows for x in row})
    counts = Counter(confusion_rows)
    with matrix_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["true\\pred", *labels_set])
        for tl in labels_set:
            writer.writerow([tl, *[counts[(tl, pl)] for pl in labels_set]])

    print(f"✅ Đã ghi {report_path}")
    print(f"✅ Đã ghi {matrix_path}")


# ── Train lại trên toàn bộ dữ liệu ─────────────────────────────────────────
MODEL_DIR = Path(DEFAULT_MODEL_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

np.savetxt(MODEL_DIR / FLATTENED_IMAGES_TXT, X)
np.savetxt(MODEL_DIR / CLASSIFICATIONS_TXT, y)

knn_full = train_knn(X, y, K)
knn_full.save(str(MODEL_DIR / KNN_MODEL))

labels_as_chars = [chr(int(v)) for v in y.ravel()]
label_counts = Counter(labels_as_chars)
write_evaluation(MODEL_DIR, accuracy, confusion_rows, label_counts, K)

print(f"✅ Đã lưu model → {MODEL_DIR / KNN_MODEL}")


## 10. Dự đoán ký tự từ thư mục ảnh mới

Thay `PREDICT_FOLDER` bằng đường dẫn thư mục chứa các file `char_*.png` cần nhận diện.


In [ ]:
PREDICT_FOLDER = Path("data/test_plate")   # ← đổi đường dẫn tại đây


def predict_folder(folder, model_dir=DEFAULT_MODEL_DIR, k=K):
    model_path = Path(model_dir) / KNN_MODEL
    if not model_path.exists():
        raise FileNotFoundError(f"Chưa có model: {model_path}. Hãy chạy bước Train trước.")

    knn = cv2.ml.KNearest_load(str(model_path))
    chars = []

    for image_path in sorted(Path(folder).glob("char_*.png")):
        sample = preprocess_char(image_path)
        _, result, _, _ = knn.findNearest(sample, k=k)
        chars.append(chr(int(result[0][0])))

    plate = "".join(chars)
    print(f"🚗 Biển số nhận diện được: {plate}")
    return plate


predict_folder(PREDICT_FOLDER)
